In [0]:
# 必要なライブラリのインストール
%pip install -U -qqqq databricks-sdk databricks-ai-search
dbutils.library.restartPython()

# 1-1. Scratch パターン：PDF のパースと AI Search 索引の作成

このノートブックでは、ボリュームの PDF を SQL の AI Functions でパースし、検索用のチャンクを作り、AI Search の索引を作って検索するまでを順に実行する。

```
ボリュームの PDF ─ai_parse_document→ parsed_documents ─ai_prep_search→ prepped_documents
                 → docs_chunks（CDF 有効）─Delta Sync→ AI Search 索引 → ANN／ハイブリッド検索
```

## このノートブックでやること
1. パラメータの設定（widget）
2. `ai_parse_document` で PDF をパースする
3. `ai_prep_search` で検索用のチャンクを作る
4. AI Search のエンドポイントと索引を作る
5. ANN とハイブリッドで検索して比べる

## 前提
- **00_データ準備** を実行し、ボリュームに PDF がある
- サーバーレスのノートブック（環境バージョン 4 で動作確認）。`ai_parse_document` は DBR 17.3 以上、`ai_prep_search` は DBR 18.2 以上が必要で、サーバーレスでは環境バージョン 3 以上を選ぶ
- `ai_prep_search` は **Beta**（2026年10月時点）。動作や出力が変わることがある
- 埋め込みモデル `databricks-qwen3-embedding-0-6b`（Foundation Model API、多言語対応）を使う

---

## 1-1-1. パラメータの設定

| widget | 内容 |
|---|---|
| `CATALOG_NAME` / `SCHEMA_NAME` / `VOLUME_NAME` | 00 で作ったもの |
| `PAGE_RANGE` | パースするページ（例 `1-40`）。空欄なら全ページ。時間を短くしたいときに使う |
| `REUSE_PARSED` | `true` なら、パース・チャンク化の結果のテーブルがすでにあるときは作り直さない（再実行で AI Functions の課金を増やさない）。PDF やオプションを変えたら `false` にする |
| `METADATA_FIELDS` | `ai_prep_search` に抽出させる文書メタデータの項目（JSON の配列） |
| `AI_SEARCH_ENDPOINT` | 使う AI Search エンドポイント。なければ Standard で作る |
| `INDEX_NAME` | 作る索引の名前（スキーマの中に作られる） |
| `EMBEDDING_MODEL_ENDPOINT` | 埋め込みモデルのエンドポイント |

In [0]:
# パラメータ設定（widget から取得）
dbutils.widgets.text("CATALOG_NAME", "main", "カタログ名")
dbutils.widgets.text("SCHEMA_NAME", "rag_handson", "スキーマ名")
dbutils.widgets.text("VOLUME_NAME", "docs", "ボリューム名")
dbutils.widgets.text("PAGE_RANGE", "", "パースするページ（例: 1-40、空欄は全ページ）")
dbutils.widgets.dropdown("REUSE_PARSED", "true", ["true", "false"], "パース・チャンク化の結果があれば再利用する")
dbutils.widgets.text("METADATA_FIELDS", '["document_title", "document_version", "publisher"]', "抽出するメタデータ項目")
dbutils.widgets.text("AI_SEARCH_ENDPOINT", "rag-handson-endpoint", "AI Search エンドポイント")
dbutils.widgets.text("INDEX_NAME", "docs_chunks_index", "AI Search 索引名")
dbutils.widgets.text("EMBEDDING_MODEL_ENDPOINT", "databricks-qwen3-embedding-0-6b", "埋め込みモデル")

CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
VOLUME_NAME = dbutils.widgets.get("VOLUME_NAME")
PAGE_RANGE = dbutils.widgets.get("PAGE_RANGE").strip()
REUSE_PARSED = dbutils.widgets.get("REUSE_PARSED") == "true"
METADATA_FIELDS = dbutils.widgets.get("METADATA_FIELDS").strip()
AI_SEARCH_ENDPOINT = dbutils.widgets.get("AI_SEARCH_ENDPOINT")
INDEX_NAME = dbutils.widgets.get("INDEX_NAME")
EMBEDDING_MODEL_ENDPOINT = dbutils.widgets.get("EMBEDDING_MODEL_ENDPOINT")

VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"
SOURCE_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.docs_chunks"
INDEX_FULLNAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.{INDEX_NAME}"

spark.sql(f"USE CATALOG `{CATALOG_NAME}`")
spark.sql(f"USE SCHEMA `{SCHEMA_NAME}`")

print(f"ボリューム: {VOLUME_PATH}")
print(f"ページ範囲: {PAGE_RANGE or '全ページ'}")
print(f"索引: {INDEX_FULLNAME}（エンドポイント: {AI_SEARCH_ENDPOINT}）")

## 1-1-2. ai_parse_document で PDF をパースする

`ai_parse_document` は、レイアウトを理解して文書を **要素**（`text`、`table`、`figure`、`section_header`、`page_header` など）に分けて返す SQL 関数である。

- 表は HTML、図は AI が書いた説明文として返る
- 要素ごとに位置（`bbox`）とページ番号が付く
- 1ファイル 500 ページ・100MB まで。`pageRange` でページを絞れる
- 課金は AI Functions（`AI_FUNCTIONS`）

ボリュームのファイルを `READ_FILES(..., format => 'binaryFile')` でバイナリとして読み、そのまま関数に渡す。結果は `VARIANT` 型の列になる。166ページの PDF では数分かかる。

In [0]:
options = {"version": "2.0"}
if PAGE_RANGE:
    options["pageRange"] = PAGE_RANGE
options_sql = "map(" + ", ".join(f"'{k}', '{v}'" for k, v in options.items()) + ")"

if REUSE_PARSED and spark.catalog.tableExists("parsed_documents"):
    print("parsed_documents があるため、パースし直さない（REUSE_PARSED=true）")
else:
    spark.sql(f"""
CREATE OR REPLACE TABLE parsed_documents AS
SELECT
  path,
  ai_parse_document(content, {options_sql}) AS parsed
FROM READ_FILES('{VOLUME_PATH}/', format => 'binaryFile')
""")

display(spark.sql("""
SELECT
  path,
  parsed:error_status AS error_status,
  size(cast(parsed:document.pages AS ARRAY<VARIANT>)) AS pages,
  size(cast(parsed:document.elements AS ARRAY<VARIANT>)) AS elements
FROM parsed_documents
"""))

### パース結果の確認

要素の種類ごとの件数を見る。`page_header` や `page_footer` も別の要素として取れていることが分かる。

In [0]:
display(spark.sql("""
SELECT
  el.value:type::STRING AS element_type,
  count(*) AS n
FROM parsed_documents,
  LATERAL variant_explode(parsed:document.elements) AS el
GROUP BY 1
ORDER BY 2 DESC
"""))

表（`table`）の要素は HTML で返る。最初の数件の中身を見る。

In [0]:
display(spark.sql("""
SELECT
  el.value:bbox[0].page_id::INT AS page_id,
  el.value:content::STRING AS table_html
FROM parsed_documents,
  LATERAL variant_explode(parsed:document.elements) AS el
WHERE el.value:type::STRING = 'table'
LIMIT 3
"""))

## 1-1-3. ai_prep_search で検索用のチャンクを作る

`ai_prep_search` は、`ai_parse_document` の出力を受け取り、RAG 向けのチャンクに分ける SQL 関数である（**Beta**）。チャンクごとに次を返す。

| フィールド | 内容 |
|---|---|
| `chunk_id` | 文書の ID とチャンクの位置から作る ID。索引の主キーに使う |
| `chunk_to_embed` | **埋め込み用のテキスト**。タイトルや見出しなどの文書メタデータ、文書の要約、元のチャンクを組み合わせたもの |
| `chunk_to_retrieve` | **表示・回答に使う原文** |
| `metadata` | `schema` オプションで指定した項目の値 |
| `pages` | チャンクが含まれるページ |

検索には文脈を補った `chunk_to_embed` を使い、LLM に渡すのは原文の `chunk_to_retrieve` にする、という使い分けができる。

`schema` オプションに項目名の配列を渡すと、文書全体からその項目を抽出して `metadata` に入れる。

In [0]:
if REUSE_PARSED and spark.catalog.tableExists("prepped_documents"):
    print("prepped_documents があるため、作り直さない（REUSE_PARSED=true）")
else:
    spark.sql(f"""
CREATE OR REPLACE TABLE prepped_documents AS
SELECT
  path,
  ai_prep_search(parsed, map('version', '2.0', 'schema', '{METADATA_FIELDS}')) AS prepped
FROM parsed_documents
""")

display(spark.sql("""
SELECT
  path,
  prepped:error_status AS error_status,
  size(cast(prepped:document.contents AS ARRAY<VARIANT>)) AS chunks
FROM prepped_documents
"""))

### チャンクをテーブルに展開する

`variant_explode` でチャンクを1行ずつに展開し、AI Search の索引の元になる Delta テーブル `docs_chunks` を作る。

- Standard エンドポイントの Delta Sync 索引は、元テーブルの **Change Data Feed（CDF）** を使って同期する。テーブル作成時に `delta.enableChangeDataFeed = true` を付ける
- ページ番号は、回答で引用できるよう `pages` 列（例 `12,13`）にまとめる

In [0]:
CHUNKS_REBUILT = False
if REUSE_PARSED and spark.catalog.tableExists("docs_chunks"):
    print("docs_chunks があるため、作り直さない（REUSE_PARSED=true）")
else:
    CHUNKS_REBUILT = True
    spark.sql("""
CREATE OR REPLACE TABLE docs_chunks
TBLPROPERTIES (delta.enableChangeDataFeed = true)
AS
SELECT
  chunk.value:chunk_id::STRING AS chunk_id,
  p.path,
  regexp_extract(p.path, '[^/]+$', 0) AS file_name,
  chunk.value:chunk_position::INT AS chunk_position,
  chunk.value:chunk_to_embed::STRING AS chunk_to_embed,
  chunk.value:chunk_to_retrieve::STRING AS chunk_to_retrieve,
  array_join(
    transform(cast(chunk.value:pages AS ARRAY<VARIANT>), x -> cast(x:page_id::INT + 1 AS STRING)),
    ','
  ) AS pages,
  to_json(chunk.value:metadata) AS metadata
FROM prepped_documents p,
  LATERAL variant_explode(p.prepped:document.contents) AS chunk
""")

display(spark.sql("SELECT count(*) AS chunks, count(DISTINCT path) AS files FROM docs_chunks"))

### chunk_to_embed と chunk_to_retrieve を比べる

同じチャンクの2つのテキストを並べる。`chunk_to_embed` には、文書のタイトルや見出し、要約などの文脈が足されていることを確かめる。

In [0]:
display(spark.sql("""
SELECT chunk_position, pages, chunk_to_embed, chunk_to_retrieve, metadata
FROM docs_chunks
ORDER BY chunk_position
LIMIT 5
"""))

## 1-1-4. AI Search のエンドポイントと索引を作る

**AI Search**（旧 Vector Search）は、Delta テーブルと同期するベクトル検索・ハイブリッド検索のサービスである。

| 部品 | 内容 |
|---|---|
| エンドポイント | 検索を提供する計算資源。**Standard**（低レイテンシ）と **Storage-optimized**（大量データ）がある。稼働中は課金される |
| 索引（Delta Sync） | Delta テーブルと同期する索引。埋め込みは Databricks が指定のモデルで計算する（managed embeddings） |

エンドポイントがなければ Standard で作る。作成には数分かかる。

> UI で作る場合：左サイドバー「コンピュート」→「AI Search」タブでエンドポイントを作り、カタログエクスプローラーで `docs_chunks` テーブルを開いて「作成」から索引を作る。

In [0]:
from databricks.ai_search.client import AISearchClient

vsc = AISearchClient(disable_notice=True)

if vsc.endpoint_exists(AI_SEARCH_ENDPOINT):
    print(f"既存のエンドポイントを使う: {AI_SEARCH_ENDPOINT}")
else:
    print(f"エンドポイントを作る（数分かかる）: {AI_SEARCH_ENDPOINT}")
    vsc.create_endpoint_and_wait(name=AI_SEARCH_ENDPOINT, endpoint_type="STANDARD")

print(vsc.get_endpoint(AI_SEARCH_ENDPOINT)["endpoint_status"])

### Delta Sync 索引を作る

- 主キー：`chunk_id`
- 埋め込みの元にする列：`chunk_to_embed`（文脈付きのテキスト）
- 同期の方式：`TRIGGERED`（元テーブルを更新したら `sync()` で反映する）
- 元テーブルを `CREATE OR REPLACE` で作り直すと、索引は元のテーブルを追えなくなる。`REUSE_PARSED=false` で作り直したときは、このセルが索引も作り直す

索引の初回作成と埋め込みの計算には数分かかる。

In [0]:
import datetime
import time

# docs_chunks を作り直した場合、既存の索引は元のテーブルを読み続けて同期できなくなるため、作り直す
if CHUNKS_REBUILT and vsc.index_exists(endpoint_name=AI_SEARCH_ENDPOINT, index_name=INDEX_FULLNAME):
    print(f"docs_chunks を作り直したため、索引を削除して作り直す: {INDEX_FULLNAME}")
    vsc.delete_index(endpoint_name=AI_SEARCH_ENDPOINT, index_name=INDEX_FULLNAME)
    while vsc.index_exists(endpoint_name=AI_SEARCH_ENDPOINT, index_name=INDEX_FULLNAME):
        time.sleep(10)

if vsc.index_exists(endpoint_name=AI_SEARCH_ENDPOINT, index_name=INDEX_FULLNAME):
    index = vsc.get_index(endpoint_name=AI_SEARCH_ENDPOINT, index_name=INDEX_FULLNAME)
    print(f"既存の索引を使う。準備ができるのを待ってから同期する: {INDEX_FULLNAME}")
    index.wait_until_ready(verbose=False, timeout=datetime.timedelta(hours=1))
    # 前回の同期がまだ動いている場合は、終わるまで待ってから同期する
    for _ in range(60):
        try:
            index.sync()
            break
        except Exception as e:
            if "not ready to sync" not in str(e):
                raise
            time.sleep(30)
else:
    print(f"索引を作る: {INDEX_FULLNAME}")
    index = vsc.create_delta_sync_index(
        endpoint_name=AI_SEARCH_ENDPOINT,
        index_name=INDEX_FULLNAME,
        source_table_name=SOURCE_TABLE,
        primary_key="chunk_id",
        pipeline_type="TRIGGERED",
        embedding_source_column="chunk_to_embed",
        embedding_model_endpoint_name=EMBEDDING_MODEL_ENDPOINT,
        columns_to_sync=["chunk_id", "file_name", "chunk_position", "chunk_to_embed", "chunk_to_retrieve", "pages"],
    )

index.wait_until_ready(verbose=False, timeout=datetime.timedelta(hours=1), wait_for_updates=True)
print(index.describe()["status"])

## 1-1-5. ANN とハイブリッドで検索して比べる

AI Search の `query_type` で検索の方式を選ぶ。

| `query_type` | 内容 |
|---|---|
| `ANN`（既定） | ベクトルの近さで探す。言い換えや意味の近さに強い |
| `hybrid` | ベクトル検索とキーワード検索（BM25）の結果を RRF で統合する。型番や用語の完全一致にも強い。多くの用途の出発点として推奨されている |

同じ質問で両方を実行し、上位の結果を比べる。`TEST_QUERY` を変えて何度か試す。

In [0]:
TEST_QUERY = "プロセス能力レベル2を達成するために必要なプロセス属性は何か"

def search(query_type: str, k: int = 5):
    res = index.similarity_search(
        query_text=TEST_QUERY,
        columns=["chunk_id", "pages", "chunk_to_retrieve"],
        num_results=k,
        query_type=query_type,
    )
    cols = [c["name"] for c in res["manifest"]["columns"]]
    rows = [dict(zip(cols, r)) for r in res["result"]["data_array"]]
    return [(query_type, i + 1, r["pages"], round(r["score"], 4), r["chunk_to_retrieve"][:300]) for i, r in enumerate(rows)]

results = search("ANN") + search("hybrid")
display(spark.createDataFrame(results, "query_type STRING, rank INT, pages STRING, score DOUBLE, text STRING"))

### 参考：SQL の vector_search 関数で検索する

SQL から `vector_search()` 関数（Public Preview）でも同じ索引を検索できる。ダッシュボードやバッチ処理から使うときに便利である。

In [0]:
display(spark.sql(f"""
SELECT pages, chunk_to_retrieve
FROM vector_search(
  index => '{INDEX_FULLNAME}',
  query_text => '{TEST_QUERY}',
  query_type => 'HYBRID',
  num_results => 3
)
"""))

## まとめ

- `ai_parse_document` で PDF を要素に分け、`ai_prep_search` で埋め込み用（`chunk_to_embed`）と表示用（`chunk_to_retrieve`）のテキストを持つチャンクを作った
- CDF を有効にした Delta テーブルから、AI Search の Delta Sync 索引を作った
- ANN とハイブリッドで検索結果を比べた

### 次のステップ
**1-2_Scratch_RAGエージェントの構築とデプロイ** で、この索引を検索ツールとして使うエージェントを作る。

### 参考リンク
- [ai_parse_document](https://docs.databricks.com/aws/ja/sql/language-manual/functions/ai_parse_document)
- [ai_prep_search](https://docs.databricks.com/aws/ja/sql/language-manual/functions/ai_prep_search)
- [AI Search](https://docs.databricks.com/aws/ja/ai-search/ai-search/)
- [AI Search の索引を検索する](https://docs.databricks.com/aws/ja/ai-search/query-ai-search)